In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from keras.preprocessing.sequence import pad_sequences
from keras_preprocessing.text import Tokenizer
from gensim.models import Word2Vec, KeyedVectors
from sklearn.feature_extraction.text import TfidfVectorizer
import string

2024-10-09 10:40:59.767908: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
data = pd.read_csv('salafi_model_data.csv')

In [3]:
# load the best1 and best2 models
best1 = tf.keras.models.load_model('best1.h5')
best2 = tf.keras.models.load_model('best2.h5')

In [4]:
# load tokenizer 
import pickle
with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

# load vactorizer 
with open('vectorizer.pickle', 'rb') as handle:
    vectorizer = pickle.load(handle)

In [5]:
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize text
    words = text.split()
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
        
    # Stem words
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
        
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]


    return ' '.join(words)

In [6]:
def predict_data(data, best1, best2, tokenizer, vectorizer, max_sequence_length):
    # Tokenize the tweet sequences for best1
    data['tweet_seq'] = tokenizer.texts_to_sequences(data['tweet'])
    
    # Ensure the sequences are padded using the correct max sequence length for best1
    tweet_seq_reshaped = pad_sequences(data['tweet_seq'], maxlen=max_sequence_length)

    # Get probability predictions from best1
    data['best1_prob'] = best1.predict(tweet_seq_reshaped).flatten()  # Flatten to ensure 1D array

    # Apply vectorizer to all rows
    tweet_vectors = vectorizer.transform(data['tweet']).toarray()  # Convert sparse matrix to dense

    # Get probability predictions from best2
    best2_probabilities = best2.predict(tweet_vectors).flatten()  # Flatten to ensure 1D array

    # Combine the probabilities using soft voting (average of both models' predictions)
    data['final_prob'] = (data['best1_prob'] + best2_probabilities) / 2

    # Convert final probabilities to binary predictions based on a threshold of 0.5
    data['final_prediction'] = (data['final_prob'] >= 0.5).astype(int)

    return data


In [7]:
data['tweet'] = data['tweet'].apply(preprocess_text)

In [8]:
max_sequence_length = max([len(seq) for seq in tokenizer.texts_to_sequences(data)])  # Calculate based on training data

predicted_data = predict_data(data, best1, best2, tokenizer, vectorizer, max_sequence_length)


125/125 [==============================] - 1s 4ms/step


In [9]:
data

,tweet,label,tweet_seq,best1_prob,final_prob,final_prediction
0,obviou follow islam howev naysay doubter alway...,Negative,"[4295, 60, 3, 322, 6658, 6659, 257, 66, 90]",0.450019,0.347510,0
1,may allah make u righteou muslimahgo goal,Negative,"[7, 1, 32, 12, 700, 5415, 772]",0.492185,0.350731,0
2,due confer weekend fiqh dua lesson postpon enc...,Negative,"[233, 518, 422, 75, 93, 10, 989, 272, 286, 597...",0.459224,0.393725,0
3,simpl fact predawn sahr meal summaris shaikh s...,Positive,"[1008, 448, 6804, 1486, 2096, 70, 786, 321]",0.354042,0.497030,0
4,muslim admit homosexu tendenc without act like...,Negative,"[5, 1503, 5603, 3316, 259, 504, 61, 20, 393, 2...",0.475086,0.374221,0
...,...,...,...,...,...,...
3966,rectifi remain ramadan abu inaaya seif tunein via,Negative,"[1658, 198, 21, 9, 5256, 1145, 1323, 78]",0.461773,0.596722,1
3967,alway ask allah moham somali islam allah musli...,Positive,"[257, 88, 1, 484, 389, 3, 1, 5, 36, 648, 400, ...",0.461773,0.616002,1
3968,live lesson saturday pm import safeguard tongu...,Positive,"[17, 10, 125, 8, 213, 1657, 914, 203, 91, 50, ...",0.523137,0.737787,1
3969,death nobl shaykh salih alluhaydan may allah g...,Positive,"[354, 365, 4, 152, 2935, 7, 1, 216, 138, 78, 203]",0.442759,0.576243,1
